In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_path = "/content/drive/MyDrive/music-success-analytics"

processed_path = project_path + "/processed"
quality_reports_path = project_path + "/quality_reports"
sql_exports_path = project_path + "/sql_exports"

In [ ]:
os.makedirs(sql_exports_path, exist_ok=True)

os.listdir(project_path)

['data', 'quality_reports', 'raw', 'processed', 'sql_exports']

In [ ]:
file_path = processed_path + "/spotify_tracks_clean.csv"

df = pd.read_csv(file_path)

df.head()

,track_id,track_name,artist_name,album_name,release_date,release_year,genre,duration_ms,duration_min,popularity,...,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label,label_type
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,2016,Pop,234194,3.90,55,...,9,-32.22,0,0.436,73.12,13000,Brazil,0,Universal Music,Label
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,2022,Metal,375706,6.26,45,...,0,-14.02,0,0.223,157.74,1000,France,1,Island Records,Label
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,2016,Rock,289191,4.82,55,...,8,-48.26,1,0.584,71.03,1000,Germany,1,XL Recordings,Label
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,2015,Pop,209484,3.49,51,...,1,-34.47,1,0.684,149.00,1000,France,0,Warner Music,Label
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,2022,Indie,127435,2.12,39,...,10,-17.84,0,0.304,155.85,2000,United States,0,Independent,Independent


In [ ]:
df.shape

(84979, 23)

In [ ]:
df.columns

Index(['track_id', 'track_name', 'artist_name', 'album_name', 'release_date',
       'release_year', 'genre', 'duration_ms', 'duration_min', 'popularity',
       'is_hit', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'instrumentalness', 'tempo', 'stream_count', 'country', 'explicit',
       'label', 'label_type'],
      dtype='object')

In [ ]:
dim_artists = (
    df[["artist_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_artists["artist_id"] = dim_artists.index + 1

dim_artists = dim_artists[["artist_id", "artist_name"]]

dim_artists.head()

,artist_id,artist_name
0,1,Noah Rhodes
1,2,Jennifer Cole
2,3,Brandon Davis
3,4,Corey Jones
4,5,Mark Diaz


In [ ]:
dim_artists.shape

(62378, 2)

In [ ]:
dim_albums = (
    df[["album_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_albums["album_id"] = dim_albums.index + 1

dim_albums = dim_albums[["album_id", "album_name"]]

dim_albums.head()

,album_id,album_name
0,1,Beautiful instead
1,2,Table
2,3,Page southern
3,4,Spring
4,5,Great prove


In [ ]:
dim_albums.shape

(43162, 2)

In [ ]:
dim_genres = (
    df[["genre"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_genres["genre_id"] = dim_genres.index + 1

dim_genres = dim_genres[["genre_id", "genre"]]

dim_genres

,genre_id,genre
0,1,Pop
1,2,Metal
2,3,Rock
3,4,Indie
4,5,Country
5,6,Classical
6,7,Hip-Hop
7,8,EDM
8,9,Reggaeton
9,10,Folk


In [ ]:
dim_countries = (
    df[["country"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_countries["country_id"] = dim_countries.index + 1

dim_countries = dim_countries[["country_id", "country"]]

dim_countries

,country_id,country
0,1,Brazil
1,2,France
2,3,Germany
3,4,United States
4,5,Australia
5,6,United Kingdom
6,7,Japan
7,8,Canada
8,9,India
9,10,Mexico


In [ ]:
dim_labels = (
    df[["label", "label_type"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_labels["label_id"] = dim_labels.index + 1

dim_labels = dim_labels[["label_id", "label", "label_type"]]

dim_labels

,label_id,label,label_type
0,1,Universal Music,Label
1,2,Island Records,Label
2,3,XL Recordings,Label
3,4,Warner Music,Label
4,5,Independent,Independent
5,6,Sony Music,Label
6,7,EMI,Label
7,8,Columbia,Label


## SQL Data Modeling

The cleaned Spotify dataset was transformed from a single flat table into a relational structure.  
Separate dimension tables were created for artists, albums, genres, countries and labels.  
This approach reduces repeated textual values and prepares the dataset for SQL-based analysis.

In [ ]:
dimension_shapes = pd.DataFrame({
    "table_name": [
        "dim_artists",
        "dim_albums",
        "dim_genres",
        "dim_countries",
        "dim_labels"
    ],
    "rows": [
        dim_artists.shape[0],
        dim_albums.shape[0],
        dim_genres.shape[0],
        dim_countries.shape[0],
        dim_labels.shape[0]
    ],
    "columns": [
        dim_artists.shape[1],
        dim_albums.shape[1],
        dim_genres.shape[1],
        dim_countries.shape[1],
        dim_labels.shape[1]
    ]
})

dimension_shapes

,table_name,rows,columns
0,dim_artists,62378,2
1,dim_albums,43162,2
2,dim_genres,12,2
3,dim_countries,10,2
4,dim_labels,8,3


In [ ]:
fact_tracks = df.copy()

fact_tracks = fact_tracks.merge(dim_artists, on="artist_name", how="left")
fact_tracks = fact_tracks.merge(dim_albums, on="album_name", how="left")
fact_tracks = fact_tracks.merge(dim_genres, on="genre", how="left")
fact_tracks = fact_tracks.merge(dim_countries, on="country", how="left")
fact_tracks = fact_tracks.merge(dim_labels, on=["label", "label_type"], how="left")

fact_tracks.head()

,track_id,track_name,artist_name,album_name,release_date,release_year,genre,duration_ms,duration_min,popularity,...,stream_count,country,explicit,label,label_type,artist_id,album_id,genre_id,country_id,label_id
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,2016,Pop,234194,3.90,55,...,13000,Brazil,0,Universal Music,Label,1,1,1,1,1
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,2022,Metal,375706,6.26,45,...,1000,France,1,Island Records,Label,2,2,2,2,2
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,2016,Rock,289191,4.82,55,...,1000,Germany,1,XL Recordings,Label,3,3,3,3,3
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,2015,Pop,209484,3.49,51,...,1000,France,0,Warner Music,Label,4,4,1,2,4
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,2022,Indie,127435,2.12,39,...,2000,United States,0,Independent,Independent,5,5,4,4,5


In [ ]:
fact_tracks = fact_tracks[
    [
        "track_id",
        "track_name",
        "artist_id",
        "album_id",
        "genre_id",
        "country_id",
        "label_id",
        "release_date",
        "release_year",
        "duration_ms",
        "duration_min",
        "popularity",
        "is_hit",
        "danceability",
        "energy",
        "key",
        "loudness",
        "mode",
        "instrumentalness",
        "tempo",
        "stream_count",
        "explicit"
    ]
]

fact_tracks.head()

,track_id,track_name,artist_id,album_id,genre_id,country_id,label_id,release_date,release_year,duration_ms,...,is_hit,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,explicit
0,TRK-BEBD53DA84E1,Agent every (0),1,1,1,1,1,2016-04-01,2016,234194,...,0,0.15,0.74,9,-32.22,0,0.436,73.12,13000,0
1,TRK-6A32496762D7,Night respond,2,2,2,2,2,2022-04-15,2022,375706,...,0,0.44,0.46,0,-14.02,0,0.223,157.74,1000,1
2,TRK-47AA7523463E,Future choice whatever,3,3,3,3,3,2016-02-23,2016,289191,...,0,0.62,0.80,8,-48.26,1,0.584,71.03,1000,1
3,TRK-25ADA22E3B06,Bad fall pick those,4,4,1,2,4,2015-10-12,2015,209484,...,0,0.78,0.98,1,-34.47,1,0.684,149.00,1000,0
4,TRK-9245F2AD996A,Husband,5,5,4,4,5,2022-07-08,2022,127435,...,0,0.74,0.18,10,-17.84,0,0.304,155.85,2000,0


In [ ]:
fact_tracks.shape

(84979, 22)

In [ ]:
fact_tracks.isna().sum().sort_values(ascending=False).head()

,0
track_id,0
track_name,0
artist_id,0
album_id,0
genre_id,0


In [ ]:
fact_tracks["track_id"].duplicated().sum()

np.int64(0)

## Fact Table Creation

The `fact_tracks` table was created by joining the cleaned Spotify dataset with the dimension tables.  
Textual attributes such as artist, album, genre, country and label were replaced by their corresponding IDs.  
This structure creates a star schema suitable for SQL analysis, with `fact_tracks` as the central fact table and multiple supporting dimension tables.

NameError: name 'music' is not defined

In [ ]:
dim_artists.to_csv(sql_exports_path + "/dim_artists.csv", index=False)
dim_albums.to_csv(sql_exports_path + "/dim_albums.csv", index=False)
dim_genres.to_csv(sql_exports_path + "/dim_genres.csv", index=False)
dim_countries.to_csv(sql_exports_path + "/dim_countries.csv", index=False)
dim_labels.to_csv(sql_exports_path + "/dim_labels.csv", index=False)
fact_tracks.to_csv(sql_exports_path + "/fact_tracks.csv", index=False)

In [ ]:
os.listdir(sql_exports_path)

['dim_artists.csv',
 'dim_albums.csv',
 'dim_genres.csv',
 'dim_countries.csv',
 'dim_labels.csv',
 'fact_tracks.csv']

In [ ]:
sql_model_summary = pd.DataFrame({
    "table_name": [
        "fact_tracks",
        "dim_artists",
        "dim_albums",
        "dim_genres",
        "dim_countries",
        "dim_labels"
    ],
    "table_type": [
        "fact_table",
        "dimension_table",
        "dimension_table",
        "dimension_table",
        "dimension_table",
        "dimension_table"
    ],
    "rows": [
        fact_tracks.shape[0],
        dim_artists.shape[0],
        dim_albums.shape[0],
        dim_genres.shape[0],
        dim_countries.shape[0],
        dim_labels.shape[0]
    ],
    "columns": [
        fact_tracks.shape[1],
        dim_artists.shape[1],
        dim_albums.shape[1],
        dim_genres.shape[1],
        dim_countries.shape[1],
        dim_labels.shape[1]
    ],
    "primary_key": [
        "track_id",
        "artist_id",
        "album_id",
        "genre_id",
        "country_id",
        "label_id"
    ]
})

sql_model_summary

,table_name,table_type,rows,columns,primary_key
0,fact_tracks,fact_table,84979,22,track_id
1,dim_artists,dimension_table,62378,2,artist_id
2,dim_albums,dimension_table,43162,2,album_id
3,dim_genres,dimension_table,12,2,genre_id
4,dim_countries,dimension_table,10,2,country_id
5,dim_labels,dimension_table,8,3,label_id


In [ ]:
sql_model_summary.to_csv(
    quality_reports_path + "/sql_model_summary.csv",
    index=False
)

## SQL Star Schema

The dataset was modeled using a simple star schema.

The central table is `fact_tracks`, which contains one row per track and stores measurable values such as popularity, streams, duration, tempo, danceability, energy and hit status.

The supporting dimension tables are:

- `dim_artists`
- `dim_albums`
- `dim_genres`
- `dim_countries`
- `dim_labels`

Each dimension table contains unique descriptive attributes and a primary key.  
The `fact_tracks` table references these dimensions through foreign keys.

In [ ]:
mysql_schema = """
CREATE DATABASE IF NOT EXISTS music_success_analytics;
USE music_success_analytics;

DROP TABLE IF EXISTS fact_tracks;
DROP TABLE IF EXISTS dim_artists;
DROP TABLE IF EXISTS dim_albums;
DROP TABLE IF EXISTS dim_genres;
DROP TABLE IF EXISTS dim_countries;
DROP TABLE IF EXISTS dim_labels;

CREATE TABLE dim_artists (
    artist_id INT PRIMARY KEY,
    artist_name VARCHAR(255)
);

CREATE TABLE dim_albums (
    album_id INT PRIMARY KEY,
    album_name VARCHAR(255)
);

CREATE TABLE dim_genres (
    genre_id INT PRIMARY KEY,
    genre VARCHAR(100)
);

CREATE TABLE dim_countries (
    country_id INT PRIMARY KEY,
    country VARCHAR(100)
);

CREATE TABLE dim_labels (
    label_id INT PRIMARY KEY,
    label VARCHAR(255),
    label_type VARCHAR(50)
);

CREATE TABLE fact_tracks (
    track_id VARCHAR(50) PRIMARY KEY,
    track_name VARCHAR(255),
    artist_id INT,
    album_id INT,
    genre_id INT,
    country_id INT,
    label_id INT,
    release_date DATE,
    release_year INT,
    duration_ms INT,
    duration_min DECIMAL(6,2),
    popularity INT,
    is_hit INT,
    danceability DECIMAL(5,3),
    energy DECIMAL(5,3),
    `key` INT,
    loudness DECIMAL(6,2),
    mode INT,
    instrumentalness DECIMAL(5,3),
    tempo DECIMAL(6,2),
    stream_count BIGINT,
    explicit INT,
    FOREIGN KEY (artist_id) REFERENCES dim_artists(artist_id),
    FOREIGN KEY (album_id) REFERENCES dim_albums(album_id),
    FOREIGN KEY (genre_id) REFERENCES dim_genres(genre_id),
    FOREIGN KEY (country_id) REFERENCES dim_countries(country_id),
    FOREIGN KEY (label_id) REFERENCES dim_labels(label_id)
);
"""

In [ ]:
schema_file_path = sql_exports_path + "/create_tables_mysql.sql"

with open(schema_file_path, "w") as file:
    file.write(mysql_schema)

schema_file_path

'/content/drive/MyDrive/music-success-analytics/sql_exports/create_tables_mysql.sql'

In [ ]:
os.listdir(sql_exports_path)

['dim_artists.csv',
 'dim_albums.csv',
 'dim_genres.csv',
 'dim_countries.csv',
 'dim_labels.csv',
 'fact_tracks.csv',
 'create_tables_mysql.sql']

In [ ]:
business_queries = """
USE music_success_analytics;

-- 1. Top genres by average popularity
SELECT
    g.genre,
    COUNT(f.track_id) AS total_tracks,
    ROUND(AVG(f.popularity), 2) AS avg_popularity,
    SUM(f.stream_count) AS total_streams,
    ROUND(AVG(f.stream_count), 0) AS avg_streams
FROM fact_tracks f
JOIN dim_genres g
    ON f.genre_id = g.genre_id
GROUP BY g.genre
ORDER BY avg_popularity DESC;


-- 2. Top genres by total streams
SELECT
    g.genre,
    COUNT(f.track_id) AS total_tracks,
    SUM(f.stream_count) AS total_streams,
    ROUND(AVG(f.stream_count), 0) AS avg_streams
FROM fact_tracks f
JOIN dim_genres g
    ON f.genre_id = g.genre_id
GROUP BY g.genre
ORDER BY total_streams DESC;


-- 3. Hit rate by genre
SELECT
    g.genre,
    COUNT(f.track_id) AS total_tracks,
    ROUND(AVG(f.is_hit) * 100, 2) AS hit_rate_percentage
FROM fact_tracks f
JOIN dim_genres g
    ON f.genre_id = g.genre_id
GROUP BY g.genre
ORDER BY hit_rate_percentage DESC;


-- 4. Independent vs label-backed performance
SELECT
    l.label_type,
    COUNT(f.track_id) AS total_tracks,
    ROUND(AVG(f.popularity), 2) AS avg_popularity,
    ROUND(AVG(f.stream_count), 0) AS avg_streams,
    ROUND(AVG(f.is_hit) * 100, 2) AS hit_rate_percentage
FROM fact_tracks f
JOIN dim_labels l
    ON f.label_id = l.label_id
GROUP BY l.label_type
ORDER BY avg_streams DESC;


-- 5. Best performing countries by average streams
SELECT
    c.country,
    COUNT(f.track_id) AS total_tracks,
    ROUND(AVG(f.popularity), 2) AS avg_popularity,
    SUM(f.stream_count) AS total_streams,
    ROUND(AVG(f.stream_count), 0) AS avg_streams,
    ROUND(AVG(f.is_hit) * 100, 2) AS hit_rate_percentage
FROM fact_tracks f
JOIN dim_countries c
    ON f.country_id = c.country_id
GROUP BY c.country
ORDER BY avg_streams DESC;


-- 6. Top labels by average streams
SELECT
    l.label,
    l.label_type,
    COUNT(f.track_id) AS total_tracks,
    ROUND(AVG(f.popularity), 2) AS avg_popularity,
    ROUND(AVG(f.stream_count), 0) AS avg_streams,
    ROUND(AVG(f.is_hit) * 100, 2) AS hit_rate_percentage
FROM fact_tracks f
JOIN dim_labels l
    ON f.label_id = l.label_id
GROUP BY l.label, l.label_type
ORDER BY avg_streams DESC;


-- 7. Audio features comparison: hit vs non-hit tracks
SELECT
    CASE
        WHEN f.is_hit = 1 THEN 'Hit'
        ELSE 'Non-Hit'
    END AS track_type,
    COUNT(f.track_id) AS total_tracks,
    ROUND(AVG(f.popularity), 2) AS avg_popularity,
    ROUND(AVG(f.stream_count), 0) AS avg_streams,
    ROUND(AVG(f.danceability), 2) AS avg_danceability,
    ROUND(AVG(f.energy), 2) AS avg_energy,
    ROUND(AVG(f.instrumentalness), 2) AS avg_instrumentalness,
    ROUND(AVG(f.tempo), 2) AS avg_tempo,
    ROUND(AVG(f.duration_min), 2) AS avg_duration_min,
    ROUND(AVG(f.explicit) * 100, 2) AS explicit_rate_percentage
FROM fact_tracks f
GROUP BY f.is_hit
ORDER BY f.is_hit;


-- 8. Tracks and average popularity by release year
SELECT
    f.release_year,
    COUNT(f.track_id) AS total_tracks,
    ROUND(AVG(f.popularity), 2) AS avg_popularity,
    SUM(f.stream_count) AS total_streams,
    ROUND(AVG(f.stream_count), 0) AS avg_streams
FROM fact_tracks f
GROUP BY f.release_year
ORDER BY f.release_year;


-- 9. Top 20 tracks by stream count
SELECT
    f.track_id,
    f.track_name,
    a.artist_name,
    g.genre,
    c.country,
    l.label,
    l.label_type,
    f.popularity,
    f.stream_count
FROM fact_tracks f
JOIN dim_artists a
    ON f.artist_id = a.artist_id
JOIN dim_genres g
    ON f.genre_id = g.genre_id
JOIN dim_countries c
    ON f.country_id = c.country_id
JOIN dim_labels l
    ON f.label_id = l.label_id
ORDER BY f.stream_count DESC
LIMIT 20;


-- 10. Genre and label type performance
SELECT
    g.genre,
    l.label_type,
    COUNT(f.track_id) AS total_tracks,
    ROUND(AVG(f.popularity), 2) AS avg_popularity,
    ROUND(AVG(f.stream_count), 0) AS avg_streams,
    ROUND(AVG(f.is_hit) * 100, 2) AS hit_rate_percentage
FROM fact_tracks f
JOIN dim_genres g
    ON f.genre_id = g.genre_id
JOIN dim_labels l
    ON f.label_id = l.label_id
GROUP BY g.genre, l.label_type
ORDER BY g.genre, avg_streams DESC;
"""

In [ ]:
business_queries_file_path = sql_exports_path + "/business_queries_mysql.sql"

with open(business_queries_file_path, "w") as file:
    file.write(business_queries)

business_queries_file_path

'/content/drive/MyDrive/music-success-analytics/sql_exports/business_queries_mysql.sql'

In [ ]:
os.listdir(sql_exports_path)

['dim_artists.csv',
 'dim_albums.csv',
 'dim_genres.csv',
 'dim_countries.csv',
 'dim_labels.csv',
 'fact_tracks.csv',
 'create_tables_mysql.sql',
 'business_queries_mysql.sql']

## Business SQL Queries

A set of SQL business queries was created to analyze the relational model directly in MySQL.

The queries cover:

- genre performance
- hit rate by genre
- independent vs label-backed releases
- country-level performance
- label performance
- hit vs non-hit audio feature comparison
- yearly release trends
- top tracks by stream count
- genre and label type combinations

These queries transform the cleaned dataset into business-oriented insights and make the project more relevant for Data Analyst and Data Engineer roles.

## SQL Modeling and Business Query Summary

The cleaned Spotify dataset was transformed into a relational star schema and loaded into MySQL.  
The model includes one central fact table, `fact_tracks`, and five dimension tables: artists, albums, genres, countries and labels.

After validating row counts and foreign key relationships, a set of business SQL queries was created to analyze genre performance, country-level performance, label performance, hit vs non-hit characteristics, explicit content, release year trends and artist-level performance.

The SQL analysis confirmed several patterns already observed during the Python EDA phase: popularity is relatively stable across categories, while stream count provides stronger differences between genres, countries, labels and artists.